In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from pathlib import Path

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")


In [2]:
# 读取CSV文件
csv_path = '/media/ubuntu/sda/Monkey/image_feature/image_features_756d_clip_vitl14_F.csv'
df = pd.read_csv(csv_path)

print(f"原始数据形状: {df.shape}")
print(f"类别列: {df['class'].nunique()} 个不同的类别")
print(f"\n前5个类别及其样本数:")
print(df['class'].value_counts().head())


原始数据形状: (22248, 760)
类别列: 1854 个不同的类别

前5个类别及其样本数:
class
zucchini     12
aardvark     12
abacus       12
accordion    12
acorn        12
Name: count, dtype: int64


In [3]:
# 提取特征列（feature_0 到 feature_755，共756维）
feature_cols = [f'feature_{i}' for i in range(756)]

# 按class分组，对每个类别的特征求平均值
class_features = df.groupby('class')[feature_cols].mean()

print(f"类别特征矩阵形状: {class_features.shape} (n_class, 756)")
print(f"类别数量: {len(class_features)}")
print(f"\n类别列表（前10个）:")
print(class_features.index.tolist()[:10])


类别特征矩阵形状: (1854, 756) (n_class, 756)
类别数量: 1854

类别列表（前10个）:
['aardvark', 'abacus', 'accordion', 'acorn', 'air_conditioner', 'air_mattress', 'air_pump', 'airbag', 'airboat', 'aircraft_carrier']


In [4]:
# 转换为numpy数组
features = class_features.values
class_names = class_features.index.tolist()

print(f"特征矩阵形状: {features.shape}")
print(f"特征值范围: [{features.min():.4f}, {features.max():.4f}]")
print(f"特征均值: {features.mean():.4f}, 标准差: {features.std():.4f}")


特征矩阵形状: (1854, 756)
特征值范围: [-11.6033, 12.4474]
特征均值: 0.0013, 标准差: 0.6310


In [5]:
# 标准化特征（可选，但通常有助于PCA和聚类）
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# PCA降维到30维
pca = PCA(n_components=30, random_state=42)
features_pca = pca.fit_transform(features_scaled)

print(f"PCA后特征形状: {features_pca.shape}")
print(f"\n前10个主成分的方差解释比例:")
for i, var in enumerate(pca.explained_variance_ratio_[:10]):
    print(f"PC{i+1}: {var:.4f} ({var*100:.2f}%)")
print(f"\n累计方差解释比例: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")


PCA后特征形状: (1854, 30)

前10个主成分的方差解释比例:
PC1: 0.0581 (5.81%)
PC2: 0.0438 (4.38%)
PC3: 0.0276 (2.76%)
PC4: 0.0234 (2.34%)
PC5: 0.0201 (2.01%)
PC6: 0.0198 (1.98%)
PC7: 0.0171 (1.71%)
PC8: 0.0164 (1.64%)
PC9: 0.0150 (1.50%)
PC10: 0.0130 (1.30%)

累计方差解释比例: 0.4323 (43.23%)


In [6]:
# K-means聚类（10类）
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features_pca)

print(f"聚类完成，共{len(np.unique(cluster_labels))}个聚类")
print(f"\n各聚类包含的类别数量:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"聚类 {label}: {count} 个类别")


聚类完成，共10个聚类

各聚类包含的类别数量:
聚类 0: 299 个类别
聚类 1: 187 个类别
聚类 2: 173 个类别
聚类 3: 184 个类别
聚类 4: 139 个类别
聚类 5: 202 个类别
聚类 6: 140 个类别
聚类 7: 161 个类别
聚类 8: 193 个类别
聚类 9: 176 个类别


In [7]:
# 列出每个聚类中的类别（每个聚类显示10个类别）
print("=" * 80)
print("每个K-means聚类中的类别列表（每个聚类显示前10个类别）:")
print("=" * 80)

# 创建一个DataFrame来存储类别和对应的聚类标签
cluster_df = pd.DataFrame({
    'class': class_names,
    'cluster': cluster_labels
})

# 按聚类标签排序
cluster_df = cluster_df.sort_values('cluster')

# 遍历每个聚类
for cluster_id in sorted(np.unique(cluster_labels)):
    cluster_classes = cluster_df[cluster_df['cluster'] == cluster_id]['class'].tolist()
    total_count = len(cluster_classes)
    display_count = min(10, total_count)
    
    print(f"\n聚类 {cluster_id} (共 {total_count} 个类别):")
    print("-" * 60)
    for i, class_name in enumerate(cluster_classes[:display_count], 1):
        print(f"  {i:2d}. {class_name}")
    if total_count > display_count:
        print(f"  ... 还有 {total_count - display_count} 个类别未显示")


每个K-means聚类中的类别列表（每个聚类显示前10个类别）:

聚类 0 (共 299 个类别):
------------------------------------------------------------
   1. watering_can
   2. map
   3. manhole
   4. metal_detector
   5. megaphone
   6. medal
   7. backgammon
   8. wheelbarrow
   9. wheel
  10. girl
  ... 还有 289 个类别未显示

聚类 1 (共 187 个类别):
------------------------------------------------------------
   1. amber
   2. aluminum_foil
   3. yo-yo
   4. wick
   5. sandpaper
   6. washcloth
   7. marker
   8. marble
   9. ashtray
  10. whoopee_cushion
  ... 还有 177 个类别未显示

聚类 2 (共 173 个类别):
------------------------------------------------------------
   1. aardvark
   2. yak
   3. zebra
   4. weasel
   5. wasp
   6. dragonfly
   7. dog
   8. eel
   9. earwig
  10. iguana
  ... 还有 163 个类别未显示

聚类 3 (共 184 个类别):
------------------------------------------------------------
   1. wedding_cake
   2. whipped_cream
   3. bacon
   4. meat
   5. meatball
   6. meatloaf
   7. appetizer
   8. applesauce
   9. marmalade
  10. marshmallow
  ... 

In [8]:
# 从每个聚类中随机抽取100个类别
np.random.seed(42)  # 设置随机种子以确保可重复性

selected_classes = []  # 存储所有被选中的类别
cluster_selected = {}  # 存储每个聚类的抽取结果
n_samples_per_cluster = 100  # 每个聚类抽取的类别数

print("=" * 80)
print(f"从每个聚类中随机抽取 {n_samples_per_cluster} 个类别:")
print("=" * 80)

# 遍历每个聚类
for cluster_id in sorted(np.unique(cluster_labels)):
    cluster_classes = cluster_df[cluster_df['cluster'] == cluster_id]['class'].tolist()
    total_count = len(cluster_classes)
    
    # 如果类别数少于100，则全部抽取；否则随机抽取100个
    if total_count <= n_samples_per_cluster:
        sampled_classes = cluster_classes
        print(f"\n聚类 {cluster_id}: 共 {total_count} 个类别，全部抽取")
    else:
        sampled_classes = np.random.choice(cluster_classes, size=n_samples_per_cluster, replace=False).tolist()
        print(f"\n聚类 {cluster_id}: 共 {total_count} 个类别，随机抽取 {n_samples_per_cluster} 个")
    
    cluster_selected[cluster_id] = sampled_classes
    selected_classes.extend(sampled_classes)
    print(f"  抽取的类别数: {len(sampled_classes)}")

print(f"\n总共抽取了 {len(selected_classes)} 个类别")


从每个聚类中随机抽取 100 个类别:

聚类 0: 共 299 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 1: 共 187 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 2: 共 173 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 3: 共 184 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 4: 共 139 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 5: 共 202 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 6: 共 140 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 7: 共 161 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 8: 共 193 个类别，随机抽取 100 个
  抽取的类别数: 100

聚类 9: 共 176 个类别，随机抽取 100 个
  抽取的类别数: 100

总共抽取了 1000 个类别


In [9]:

selected_df = pd.DataFrame({
    'class': selected_classes
})

selected_df = selected_df.merge(cluster_df, on='class', how='left')


In [10]:

OBJECT_IMAGES_DIR = "/media/ubuntu/sda/visual_stimuli_pattern/things/object_images_sample10"
OUTPUT_CSV = "/media/ubuntu/sda/visual_stimuli_pattern/things/visual_stimuli_sequence_1000classes.csv"

# 设置随机种子
np.random.seed(42)
random.seed(42)

print("=" * 80)
print("生成刺激序列：从1000个类别中获取所有图片")
print("=" * 80)
print(f"目标目录: {OBJECT_IMAGES_DIR}")
print(f"输出文件: {OUTPUT_CSV}")
print(f"抽取的类别数: {len(selected_classes)}")
print(f"预期每类图片数: 10张（从object_images_sample10目录）")
print(f"预期总图片数: {len(selected_classes) * 10}")


生成刺激序列：从1000个类别中获取所有图片
目标目录: /media/ubuntu/sda/visual_stimuli_pattern/things/object_images_sample10
输出文件: /media/ubuntu/sda/visual_stimuli_pattern/things/visual_stimuli_sequence_1000classes.csv
抽取的类别数: 1000
预期每类图片数: 10张（从object_images_sample10目录）
预期总图片数: 10000


In [11]:
# 收集每个类别的所有图片（从object_images_sample10，每类应该有10张）
print("\n正在收集图片...")
category_images = {}
total_images_collected = 0
missing_categories = []
categories_with_different_counts = []

for idx, class_name in enumerate(selected_classes, 1):
    category_path = os.path.join(OBJECT_IMAGES_DIR, class_name)
    
    if not os.path.exists(category_path) or not os.path.isdir(category_path):
        missing_categories.append(class_name)
        continue
    
    # 收集该类别下的所有图片
    all_images = []
    for img_file in sorted(os.listdir(category_path)):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(category_path, img_file)
            all_images.append(img_path)
    
    if len(all_images) == 0:
        missing_categories.append(class_name)
        continue
    
    # 使用所有图片（object_images_sample10中每类应该有10张）
    category_images[class_name] = all_images
    total_images_collected += len(all_images)
    
    # 记录图片数不是10的类别
    if len(all_images) != 10:
        categories_with_different_counts.append((class_name, len(all_images)))
    
    # 每处理100个类别显示一次进度
    if idx % 100 == 0:
        print(f"  已处理 {idx}/{len(selected_classes)} 个类别...")

print(f"\n收集完成！")
print(f"成功处理的类别数: {len(category_images)}")
print(f"总图片数: {total_images_collected}")

if missing_categories:
    print(f"\n警告: {len(missing_categories)} 个类别未找到或没有图片")
    if len(missing_categories) <= 10:
        print(f"  缺失的类别: {missing_categories}")
    else:
        print(f"  前10个缺失的类别: {missing_categories[:10]}")

if categories_with_different_counts:
    print(f"\n提示: {len(categories_with_different_counts)} 个类别的图片数不是10张")
    if len(categories_with_different_counts) <= 10:
        for cat, count in categories_with_different_counts:
            print(f"  {cat}: {count} 张")



正在收集图片...
  已处理 100/1000 个类别...
  已处理 200/1000 个类别...
  已处理 300/1000 个类别...
  已处理 400/1000 个类别...
  已处理 500/1000 个类别...
  已处理 600/1000 个类别...
  已处理 700/1000 个类别...
  已处理 800/1000 个类别...
  已处理 900/1000 个类别...
  已处理 1000/1000 个类别...

收集完成！
成功处理的类别数: 1000
总图片数: 10000


In [12]:
# 创建所有图片的列表
all_image_trials = []

for class_name, images in category_images.items():
    for img_path in images:
        all_image_trials.append({
            'image_path': img_path,
            'category': class_name
        })

print(f"创建的trial总数: {len(all_image_trials)}")

# 随机打乱
print("\n正在随机打乱顺序...")
np.random.shuffle(all_image_trials)
print("打乱完成！")


创建的trial总数: 10000

正在随机打乱顺序...
打乱完成！


In [13]:
# 保存为CSV文件
print(f"\n正在保存到: {OUTPUT_CSV}")

# 创建DataFrame
data = {
    'stimulus_number': list(range(1, len(all_image_trials) + 1)),
    'image_path': [trial['image_path'] for trial in all_image_trials],
    'category': [trial['category'] for trial in all_image_trials]
}

df_sequence = pd.DataFrame(data)

# 确保输出目录存在
output_dir = os.path.dirname(OUTPUT_CSV)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 保存CSV
df_sequence.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"✓ 序列已保存到: {OUTPUT_CSV}")
print(f"总刺激次数: {len(all_image_trials)}")
print(f"\n前10行预览:")
print(df_sequence.head(10))
print(f"\n后10行预览:")
print(df_sequence.tail(10))



正在保存到: /media/ubuntu/sda/visual_stimuli_pattern/things/visual_stimuli_sequence_1000classes.csv
✓ 序列已保存到: /media/ubuntu/sda/visual_stimuli_pattern/things/visual_stimuli_sequence_1000classes.csv
总刺激次数: 10000

前10行预览:
   stimulus_number                                         image_path  \
0                1  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
1                2  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
2                3  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
3                4  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
4                5  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
5                6  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
6                7  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
7                8  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
8                9  /media/ubuntu/sda/visual_stimuli_pattern/thing...   
9               10  /media/ubuntu/sda/visual_stimuli_p

In [14]:
# 统计信息
print("\n统计信息:")
print("=" * 80)
print(f"总刺激次数: {len(all_image_trials)}")
print(f"类别数量: {df_sequence['category'].nunique()}")

# 统计每个类别的图片数
category_counts = df_sequence['category'].value_counts().sort_index()
print(f"\n每个类别的图片数统计:")
print(f"  最小: {category_counts.min()}")
print(f"  最大: {category_counts.max()}")
print(f"  平均: {category_counts.mean():.2f}")

# 检查是否有相邻的相同图片或相同类别
print(f"\n检查序列有效性...")
adjacent_same_image = 0
adjacent_same_category = 0

for i in range(len(all_image_trials) - 1):
    if all_image_trials[i]['image_path'] == all_image_trials[i+1]['image_path']:
        adjacent_same_image += 1
    if all_image_trials[i]['category'] == all_image_trials[i+1]['category']:
        adjacent_same_category += 1

print(f"  相邻相同图片: {adjacent_same_image} 处")
print(f"  相邻相同类别: {adjacent_same_category} 处")

if adjacent_same_image == 0 and adjacent_same_category == 0:
    print("  ✓ 序列有效：没有相邻的相同图片或相同类别")
else:
    print("  ⚠ 警告：序列包含相邻的相同图片或相同类别")



统计信息:
总刺激次数: 10000
类别数量: 1000

每个类别的图片数统计:
  最小: 10
  最大: 10
  平均: 10.00

检查序列有效性...
  相邻相同图片: 0 处
  相邻相同类别: 5 处
  ⚠ 警告：序列包含相邻的相同图片或相同类别
